# Part 3: Softmax Regression

In the second part of the lab we saw how to make a linear binary classifier using logistic regression. In this  part of the lab we'll turn our attention to multi-class classification.

Softmax regression (or multinomial logistic regression) is a generalisation of logistic regression to the case where we want to handle multiple classes. In logistic regression we assumed that the labels were binary: $y_i\in \{0,1\}$. We used such a classifier to distinguish between two kinds of hand-written digits. Softmax regression allows us to handle $y_i \in \{1,\dots,K\}$ where $K$ is the number of classes.

Recall that in logistic regression, we had a training set $\{(\mathbf{x}_1,y_1),\dots,(\mathbf{x}_m,y_m)\}$ of $m$ labeled examples, where the input features are $\mathbf{x}_i \in \mathbb{R}^n$. In logistic regression, our hypothesis took the form:

\begin{align}
h_\theta(\mathbf{x}) &= \frac{1}{1 + \exp(-\mathbf{x}^\top\theta)} \equiv \sigma(\mathbf{x}^\top\theta)
\end{align}

and the model parameters $\theta$ were trained to minimise the cost function

\begin{align}
J(\theta) & = -\sum_i y_i \log(\sigma(\mathbf{x}_i^\top\theta)) + (1-y_i) \log(1-\sigma(\mathbf{x}_i^\top\theta))
\end{align}

In the softmax regression setting, we are interested in multi-class classification, and so the label $y$
 can take on $K$ different values, rather than only two. Thus, in our training set $\{(\mathbf{x}_1,y_1),\dots,(\mathbf{x}_m,y_m)\}$, we now have that $y_i \in \{1,\dots,K\}$.

Given a test input $\mathbf{x}$, we want our hypothesis to estimate the probability that $P(y=k|\mathbf{x})$ for each value of $k=1,\dots,K$. That is to say, we want to estimate the probability of the class label taking on each of the $K$ different possible values. Thus, our hypothesis will output a $K$-dimensional vector (whose elements sum to 1) giving us our $K$ estimated probabilities. Concretely, our hypothesis $h_\theta(\mathbf{x})$ takes the form:

\begin{align}
h_\theta(\mathbf{x}) =
\begin{bmatrix}
P(y = 1 | \mathbf{x}; \theta) \\
P(y = 2 | \mathbf{x}; \theta) \\
\vdots \\
P(y = K | \mathbf{x}; \theta)
\end{bmatrix}
=
\frac{1}{ \sum_{j=1}^{K}{\exp(\theta^{(j)\top} \mathbf{x}) }}
\begin{bmatrix}
\exp(\theta^{(1)\top} \mathbf{x} ) \\
\exp(\theta^{(2)\top} \mathbf{x} ) \\
\vdots \\
\exp(\theta^{(K)\top} \mathbf{x} ) \\
\end{bmatrix}
\end{align}

Here $\theta^{(1)},\theta^{(2)},\dots,\theta^{(K)} \in \mathbb{R}^n$ are the parameters of our model. Notice that the term $\frac{1}{\sum_{j=1}^K exp(\theta^{(j)\top} \mathbf{x})}$ normalizes the distribution, so that it sums to one.

For convenience, we will also write $\theta$ to denote all the parameters of our model. When you implement softmax regression, it is usually convenient to represent $\theta$ as a $n$-by-$K$ matrix obtained by concatenating $\theta^{(1)},\theta^{(2)},\dots,\theta^{(K)}$ into columns, so that

\begin{align}
\theta = \left[\begin{array}{cccc}| & | & | & | \\
\theta^{(1)} & \theta^{(2)} & \cdots & \theta^{(K)} \\
| & | & | & |
\end{array}\right].
\end{align}


## Cost Function

We now describe the cost function that we’ll use for softmax regression. In the equation below, $1\{\cdot\}$
 is an "indicator function", such that $1\{\mathrm{a true statement}\}=1$, and $1\{\mathrm{a false statement}\}=0$. For example, $1\{2+2=4\}$ evaluates to $1$; whereas $1\{1+1=5\}$ evaluates to $0$. Our cost function will be:

\begin{align}
J(\theta) = - \left[ \sum_{i=1}^{m} \sum_{k=1}^{K}  1\left\{y_{i} = k\right\} \log \frac{\exp(\theta^{(k)\top} \mathbf{x}_i)}{\sum_{j=1}^K \exp(\theta^{(j)\top} \mathbf{x}_i)}\right]
\end{align}

Notice that this generalises the logistic regression cost function, which could also have been written:

\begin{align}
J(\theta) &= - \left[ \sum_{i=1}^m   (1-y^{(i)}) \log (1-h_\theta(\mathbf{x}_i)) + y^{(i)} \log h_\theta(\mathbf{x}_i) \right] \\
&= - \left[ \sum_{i=1}^{m} \sum_{k=0}^{1} 1\left\{y^{(i)} = k\right\} \log P(y^{(i)} = k | \mathbf{x}_i ; \theta) \right]
\end{align}


The softmax cost function is similar, except that we now sum over the $K$ different possible values of the class label. Note also that in softmax regression, we have that

\begin{equation}
P(y_i = k | \mathbf{x}_i ; \theta) = \frac{\exp(\theta^{(k)\top} \mathbf{x}_i)}{\sum_{j=1}^K \exp(\theta^{(j)\top} \mathbf{x}_i) }
\end{equation}

We cannot solve for the minimum of $J(\theta)$ analytically, and thus we'll resort to using gradient descent as before. Taking derivatives, one can show that the gradient is:

\begin{align}
\nabla_{\theta^{(k)}} J(\theta) = - \sum_{i=1}^{m}{ \left[ \mathbf{x}_i \left( 1\{ y_i = k\}  - P(y_i = k | \mathbf{x}_i; \theta) \right) \right]  }
\end{align}

Armed with this formula for the derivative, one can then use it directly with a gradient descent solver (or any other 1st-order gradient based optimiser).

__Use the code box below to complete the implementation of the functions that return the gradients of the softmax loss function, $\nabla_{\theta^{(k)}} J(\theta) \,\, \forall k$ and the loss function itself, $J(\theta)$:__

In [3]:
import torch

# we wouldn't normally do this, but for this lab we want to work in double precision
# as we'll need the numerical accuracy later on for doing checks on our gradients:
torch.set_default_dtype(torch.float64)

def softmax_regression_loss_grad(Theta, X, y):
    '''Implementation of the gradient of the softmax loss function.

    Theta is the matrix of parameters, with the parameters of the k-th class in the k-th column
    X contains the data vectors (one vector per row)
    y is a column vector of the targets
    '''
    num_classes = Theta.shape[1]
    indicator = (y == torch.arange(num_classes)).to(Theta.dtype)

    # 2.a predictions (num_samples, num_classes)
    Z = X @ Theta

    # 2.b softmax probabilities (subtract row max for numerical stability)
    Z = Z - Z.max(dim=1, keepdim=True)[0]
    expZ = torch.exp(Z)
    P = expZ / expZ.sum(dim=1, keepdim=True)

    # 2.c indicator minus probabilities
    diff = indicator - P

    # 3. grad (num_features, num_classes)
    grad = -X.t() @ diff
    return grad # num_features by num_classes to match shape of Theta


def softmax_regression_loss(Theta, X, y):
    '''Implementation of the softmax loss function.

    Theta is the matrix of parameters, with the parameters of the k-th class in the k-th column
    X contains the data vectors (one vector per row)
    y is a column vector of the targets
    '''
    Z = X @ Theta
    Z = Z - Z.max(dim=1, keepdim=True)[0]
    log_probs = Z - torch.log(torch.exp(Z).sum(dim=1, keepdim=True))
    loss = -log_probs.gather(1, y.long()).sum()

    return loss

__Use the following code block to confirm that your implementation is correct using gradient checking. If there are problems with your gradient or loss, go back and fix them!:__

In [4]:
 # from torch.autograd import gradcheck
from random import randrange

def grad_check(f, x, analytic_grad, num_checks=10, h=1e-3):
    sum_error = 0
    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape]) #randomly sample value to change

        oldval = x[ix].item()
        x[ix] = oldval + h # increment by h
        fxph = f(x) # evaluate f(x + h)
        x[ix] = oldval - h # increment by h
        fxmh = f(x) # evaluate f(x - h)
        x[ix] = oldval # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (abs(grad_numerical) + abs(grad_analytic) + 1e-8)
        sum_error += rel_error
        print('numerical: %f\tanalytic: %f\trelative error: %e' % (grad_numerical, grad_analytic, rel_error))
    return sum_error / num_checks

# Create some test data:
num_classes = 10
features_dim = 20
num_items = 100
Theta = torch.randn((features_dim, num_classes))
X = torch.randn((num_items,features_dim))
y = torch.randint(0, num_classes, (num_items, 1))

# compute the analytic gradient
grad = softmax_regression_loss_grad(Theta, X, y)

# run the gradient checker
grad_check(lambda th: softmax_regression_loss(th, X, y), Theta, grad)

numerical: -1.052831	analytic: -1.052831	relative error: 1.353345e-07
numerical: -4.632786	analytic: -4.632786	relative error: 1.176583e-08
numerical: 2.452875	analytic: 2.452875	relative error: 2.448025e-08
numerical: -4.330129	analytic: -4.330129	relative error: 1.172951e-08
numerical: -1.142600	analytic: -1.142600	relative error: 1.985955e-08
numerical: -0.318501	analytic: -0.318500	relative error: 5.409885e-07
numerical: 4.852293	analytic: 4.852293	relative error: 3.479193e-09
numerical: 1.748756	analytic: 1.748756	relative error: 1.095587e-08
numerical: 3.599014	analytic: 3.599014	relative error: 2.355899e-08
numerical: 11.345034	analytic: 11.345033	relative error: 5.033012e-09


tensor(7.8719e-08)

In [5]:
Theta = torch.Tensor([[1, 0], [0, 1]])
X = torch.Tensor([[1, 0], [0, 1]])
y = torch.LongTensor([[0], [1]])
assert torch.abs(softmax_regression_loss(Theta, X, y) - 0.6265) < 0.0001
grad = softmax_regression_loss_grad(Theta, X, y)
assert torch.allclose(torch.abs(grad/0.2689), torch.ones_like(grad), atol=0.001)


## Training Softmax regression with gradient descent on real data

We'll now try gradient descent with our softmax regression using the digits dataset. As before, when we looked at logistic regression, we load the data and create test and training sets. Note that this time we'll use all the classes:

In [6]:
from sklearn.datasets import load_digits

X, y = (torch.Tensor(z) for z in load_digits(n_class=10, return_X_y=True)) #convert to pytorch Tensors
X = torch.cat((X, torch.ones((X.shape[0], 1))), 1) # append a column of 1's to the X's
X /= 255
y = y.reshape(-1, 1) # reshape y into a column vector
y = y.type(torch.LongTensor)

# We're also going to break the data into a training set for computing the regression parameters
# and a test set to evaluate the predictive ability of those parameters
perm = torch.randperm(y.shape[0])
X_train = X[perm[0:260], :]
y_train = y[perm[0:260]]
X_test = X[perm[260:], :]
y_test = y[perm[260:]]

We now define a simple gradient descent loop to train the model:

In [7]:
alpha = 0.1
theta_gd = torch.rand((X_train.shape[1], 10))

for e in range(0, 1000):
    gr = softmax_regression_loss_grad(theta_gd, X_train, y_train)
    theta_gd -= alpha * gr
    if e%100 == 0:
        print("Training Loss: ", softmax_regression_loss(theta_gd, X_train, y_train))

# Compute the accuracy of the test set
proba = torch.softmax(X_test @ theta_gd, 1)
print(float((proba.argmax(1)-y_test[:,0]==0).sum()) / float(proba.shape[0]))
print()


Training Loss:  tensor(593.8891)
Training Loss:  tensor(277.5202)
Training Loss:  tensor(181.5785)
Training Loss:  tensor(138.7211)
Training Loss:  tensor(114.4017)
Training Loss:  tensor(98.5456)
Training Loss:  tensor(87.2714)
Training Loss:  tensor(78.7718)
Training Loss:  tensor(72.0884)
Training Loss:  tensor(66.6639)
0.9355888093689004



Running the above, you should observe that the training loss decreases over time. The final accuracy on the test set is also printed and should be around 90% (it will depend on the particular training/test splits you generated as well as the initial parameters for the softmax).

# Overparameterisation in softmax regression

Softmax regression has an unusual property that it has a "redundant" set of parameters. To explain what this means, suppose we take each of our parameter vectors $\theta^{(j)}$, and subtract some fixed vector $\psi$ from it, so that every $\theta^{(j)}$ is now replaced with $\theta^{(j)}−\psi$ (for every $j=1,\dots,k$). Our hypothesis now estimates the class label probabilities as

\begin{align}
P(y^{(i)} = k | x^{(i)} ; \theta)
&= \frac{\exp((\theta^{(k)}-\psi)^\top x^{(i)})}{\sum_{j=1}^K \exp( (\theta^{(j)}-\psi)^\top x^{(i)})}  \\
&= \frac{\exp(\theta^{(k)\top} x^{(i)}) \exp(-\psi^\top x^{(i)})}{\sum_{j=1}^K \exp(\theta^{(j)\top} x^{(i)}) \exp(-\psi^\top x^{(i)})} \\
&= \frac{\exp(\theta^{(k)\top} x^{(i)})}{\sum_{j=1}^K \exp(\theta^{(j)\top} x^{(i)})}.
\end{align}

__In other words, subtracting $\psi$ from every $\theta^{(j)}$ does not affect our hypothesis’ predictions at all!__ This shows that softmax regression’s parameters are "redundant". More formally, we say that our softmax model is "overparameterised" meaning that for any hypothesis we might fit to the data, there are multiple parameter settings that give rise to exactly the same hypothesis function $h_\theta$ mapping from inputs $\mathbf{x}$ to the predictions.

Further, if the cost function $J(\theta)$ is minimized by some setting of the parameters $(\theta^{(1)},\theta^{(2)},\dots,\theta^{(k)})$, then it is also minimised by $\theta^{(1)}-\psi,\theta^{(2)}-\psi,\dots,\theta^{(k)}-\psi)$ for any value of $\psi$. Thus, the minimiser of $J(\theta)$ is not unique.

(Interestingly, $J(\theta)$ is still convex, and thus gradient descent will not run into local optima problems. The Hessian is however singular/non-invertible, which causes a straightforward implementation of Newton's method (a second-order optimiser) to run into numerical problems.)

Notice also that by setting $\psi=\theta^{(K)}$, one can always replace $\theta^{(K)}$ with $\theta^{(K)}-\psi=\mathbf{0}$ (the vector of all $0$’s), without affecting the hypothesis. Thus, one could "eliminate" the vector of parameters $\theta^{(K)}$ (or any other $\theta^{(k)}$, for any single value of $k$), without harming the representational power of our hypothesis. Indeed, rather than optimising over the $K \cdot n$ parameters $(\theta^{(1)},\theta^{(2)},\dots,\theta^{(k)})$  (where $\theta^{(k)} \in \mathbb{R}^n$, one can instead set $\theta^{(K)}=\mathbf{0}$  and optimize only with respect to the $(K-1) \cdot n$ remaining parameters.

__Use the following block to implement the softmax gradients for the case where the final column of the parameters theta is fixed to be zero:__

In [8]:
import torch

def softmax_regression_loss_grad_0(Theta, X, y):
    '''Implementation of the gradient of the softmax loss function, with the parameters of the
    last class fixed to be zero.

    Theta is the matrix of parameters, with the parameters of the k-th class in the k-th column;
            K-1 classes are included, and the parameters of the last class are implicitly zero.
    X contains the data vectors (one vector per row)
    y is a column vector of the targets
    '''

    # add the missing column of zeros:
    Theta = torch.cat((Theta, Theta.new_zeros((Theta.shape[0], 1))), 1)

    grad = softmax_regression_loss_grad(Theta, X, y)

    # remove the last column from the gradients
    grad = grad[0:grad.shape[0], 0:grad.shape[1]-1]
    return grad

In [9]:
Theta = torch.Tensor([[1, 0], [0, 0]])
X = torch.Tensor([[1, 0], [0, 1]])
y = torch.LongTensor([[0], [1]])

grad = softmax_regression_loss_grad(Theta, X, y)
grad0 = softmax_regression_loss_grad_0(Theta[:,0:grad.shape[1]-1], X, y)
assert torch.allclose(grad[:,0:grad.shape[1]-1], grad0)


Finally, we can run gradient descent with our reduced parameter gradient function, and confirm that the results are similar to before:

In [10]:
alpha = 0.1
theta_gd = torch.rand((X_train.shape[1], 9))

for e in range(1000):
    gr = softmax_regression_loss_grad_0(theta_gd, X_train, y_train)
    theta_gd -= alpha * gr

theta_gd = torch.cat((theta_gd, theta_gd.new_zeros((theta_gd.shape[0], 1))), 1)
proba = torch.softmax(X_test @ theta_gd, 1)
manual_accuracy = (proba.argmax(1) == y_test[:, 0]).double().mean().item()
print("Hand-crafted softmax-regression accuracy:", manual_accuracy)


Hand-crafted softmax-regression accuracy: 0.9316851008458035


## The same model using `nn.Linear`

Our hand-crafted model computes $\mathbf X\Theta$. A PyTorch `nn.Linear` layer performs the same affine operation, storing its weight matrix in the transposed shape `(num_classes, num_features)` and optionally storing a separate bias vector.

The code below uses the original 64 image features, rescales their intensities to $[0,1]$, and lets `nn.Linear` represent the bias directly. `CrossEntropyLoss` combines the softmax and negative log-likelihood in a numerically stable form, so the model should return logits rather than probabilities.

__Task__: Run the code and compare its test accuracy with the hand-crafted implementation. Notice the familiar `zero_grad()`, `backward()`, and `step()` training pattern.

In [11]:
import torch.nn as nn

torch.manual_seed(0)

# The final column in X is the bias feature appended above. nn.Linear can
# represent that bias directly, so pass it only the original image features.
feature_scale = X_train[:, :-1].amax()
X_train_features = X_train[:, :-1] / feature_scale
X_test_features = X_test[:, :-1] / feature_scale

linear_model = nn.Linear(
    in_features=X_train_features.shape[1],
    out_features=10,
    bias=True,
)
loss_function = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(linear_model.parameters(), lr=0.1)

for epoch in range(1000):
    optimiser.zero_grad()
    logits = linear_model(X_train_features)
    loss = loss_function(logits, y_train[:, 0])
    loss.backward()
    optimiser.step()

with torch.no_grad():
    test_logits = linear_model(X_test_features)
    linear_accuracy = (
        test_logits.argmax(1) == y_test[:, 0]
    ).double().mean().item()

    # nn.Linear is still just a matrix multiplication plus a bias.
    explicit_logits = (
        X_test_features[:5] @ linear_model.weight.t()
        + linear_model.bias
    )
    torch.testing.assert_close(test_logits[:5], explicit_logits)

print("Hand-crafted accuracy:", manual_accuracy)
print("nn.Linear accuracy:     ", linear_accuracy)


Hand-crafted accuracy: 0.9316851008458035
nn.Linear accuracy:      0.936890045543266


__Task__: Briefly compare the two implementations.

- How do the shapes of `theta_gd` and `linear_model.weight` relate?
- What code did `nn.Linear`, autograd, and the optimiser remove?
- Why does `CrossEntropyLoss` take logits rather than softmax probabilities?
- Are the two test accuracies meaningfully different? Remember that different initial parameters and optimisation formulations need not produce identical weights.

**Shapes:** `theta_gd` ends up as (65, 10) once the zero column is added back on. That's one column per class, and the last row is the bias because of the column of 1s in X. `linear_model.weight` is (10, 64), so it's basically the transpose of the first 64 rows of `theta_gd`. The bias is kept separately in `linear_model.bias` with shape (10,), so there's no need for the extra column of 1s.

**Code removed:** Quite a lot. `nn.Linear` does the matrix multiplication and the bias. Autograd works out the gradients, so the softmax gradient doesn't have to be derived and coded by hand and then checked with the gradient checker. The optimiser does the `theta -= alpha * grad` update. `CrossEntropyLoss` replaces the softmax and loss function written above.

**Why logits:** Doing the softmax first and then taking the log can be unstable. If a probability underflows to 0, the log becomes −inf. Giving it logits means PyTorch can compute log-softmax directly using the log-sum-exp trick (subtracting the max logit first), which avoids this. The gradient also comes out nicely as softmax(z) − one-hot(y).

**Accuracies:** The hand-crafted version (with the last column fixed to zero) got 93.2%, and nn.Linear got 93.7%. The full-parameter version from earlier got 93.6%, so all three are pretty much the same. A 0.5% difference is only about 8 images out of the ~1500 in the test set, so it isn't really meaningful.

The two setups aren't identical anyway, so the weights wouldn't be expected to match:
- The initialisation is different: `torch.rand` in [0, 1] vs PyTorch's small default for nn.Linear.
- The scaling is different: dividing by 255 vs dividing by the max pixel value.
- The hand-crafted loss is a sum, while `CrossEntropyLoss` uses the mean, which changes the effective step size.
- The hand-crafted one fixes the last class to zero, but nn.Linear keeps all 10 columns.

Since softmax is overparameterised, keeping all 10 columns doesn't change what the model can represent. With only one random train/test split, small differences like this are most likely due to the split and initialisation rather than one method being better.